In [1]:
import pandas as pd
import pickle
import numpy as np
import talib

In [2]:
with open('../../data/nifty_data.pkl', 'rb') as file:
    nifty_data = pickle.load(file)

In [3]:
nifty_data.tail()

,Open,High,Low,Close,Volume,Dividends,Stock Splits
Date,,,,,,,
2025-09-15 00:00:00+05:30,25118.900391,25138.449219,25048.750000,25069.199219,185400,0.0,0.0
2025-09-16 00:00:00+05:30,25073.599609,25261.400391,25070.449219,25239.099609,240100,0.0,0.0
2025-09-17 00:00:00+05:30,25276.599609,25346.500000,25275.349609,25330.250000,268900,0.0,0.0
2025-09-18 00:00:00+05:30,25441.050781,25448.949219,25329.750000,25423.599609,272200,0.0,0.0
2025-09-19 00:00:00+05:30,25410.199219,25428.750000,25286.300781,25327.050781,380400,0.0,0.0


In [27]:
df = nifty_data.copy()

In [28]:
df["log_return"] = np.log(df["Close"]).diff()
for n in [5, 20, 60]:
    df[f"mom_{n}"] = df["Close"].pct_change(n)
for n in [5, 20, 60]:
    df[f"vol_{n}"] = df["log_return"].rolling(n).std()
for n in [20, 60]:
    sma = df["Close"].rolling(n).mean()
    df[f"ma_dist_{n}"] = (df["Close"] - sma) / sma
df["trend_strength"] = (
    df["mom_20"].abs() / (df["vol_20"] + 1e-8)
)
rolling_max = df["Close"].rolling(60).max()
df["drawdown_60"] = df["Close"] / rolling_max - 1
def downside_vol(series, window):
    return (
        series.clip(upper=0)
        .rolling(window)
        .std()
    )

df["down_vol_20"] = downside_vol(df["log_return"], 20)
df["atr_14"] = talib.ATR(
    df["High"],
    df["Low"],
    df["Close"],
    timeperiod=14
)
df["adx_14"] = talib.ADX(
    df["High"],
    df["Low"],
    df["Close"],
    timeperiod=14
)
df["rsi_14"] = talib.RSI(df["Close"], timeperiod=14) / 100.0


In [29]:
FEATURES = [
    "Close",
    "log_return",
    "mom_5",
    "mom_20",
    "mom_60",
    "vol_5",
    "vol_20",
    "vol_60",
    "ma_dist_20",
    "drawdown_60",
    "trend_strength",
]
FEATURES += ["down_vol_20", "atr_14"]


In [30]:
df_feat = df[FEATURES].dropna()

In [31]:
temp_copy = df_feat.copy()
df_feat = df_feat.drop("Close", axis=1)

In [32]:
mean = df_feat.mean()
std = df_feat.std()

df_feat = (df_feat - mean) / (std + 1e-8)

In [33]:
df_feat

,log_return,mom_5,mom_20,mom_60,vol_5,vol_20,vol_60,ma_dist_20,drawdown_60,trend_strength,down_vol_20,atr_14
Date,,,,,,,,,,,,
2007-12-12 00:00:00+05:30,0.752306,1.193775,0.488157,3.243550,-0.062468,0.656037,1.243965,1.678359,0.771493,-0.538049,0.758134,0.026737
2007-12-13 00:00:00+05:30,-1.297054,0.523666,0.274739,2.886924,0.540508,0.725334,1.272322,1.102040,0.501687,-0.811441,0.832179,0.028537
2007-12-14 00:00:00+05:30,-0.157005,0.349637,0.260333,2.371645,0.563353,0.726302,1.189111,1.008404,0.473960,-0.828531,0.828299,-0.049847
2007-12-17 00:00:00+05:30,-3.541012,-1.126930,-0.518713,1.795082,2.012284,1.172097,1.351685,-0.387970,-0.247747,-0.934620,1.546744,0.116650
2007-12-18 00:00:00+05:30,-0.487653,-2.066313,-0.257031,1.513815,1.360850,1.090683,1.341034,-0.561497,-0.340260,-1.187793,1.459776,0.130691
...,...,...,...,...,...,...,...,...,...,...,...,...
2025-09-15 00:00:00+05:30,-0.162174,0.338136,0.157494,-0.145729,-0.936779,-0.796803,-0.896946,0.187161,0.407300,-0.242533,-0.641192,0.516966
2025-09-16 00:00:00+05:30,0.493499,0.439156,0.102906,-0.203137,-0.856567,-0.830974,-0.924752,0.375358,0.516122,-0.393707,-0.641192,0.530114
2025-09-17 00:00:00+05:30,0.251693,0.418701,0.093366,-0.115348,-0.862160,-0.833557,-0.929980,0.466397,0.574505,-0.425718,-0.641192,0.459025


In [34]:
df_feat.describe().T

,count,mean,std,min,25%,50%,75%,max
log_return,4358.0,-8.152165e-18,0.999999,-10.700073,-0.430426,0.023721,0.475403,12.516024
mom_5,4358.0,1.956520e-17,1.000000,-6.699967,-0.483531,0.046414,0.534421,7.337880
mom_20,4358.0,-9.782598e-18,1.000000,-6.685344,-0.481925,0.037313,0.547813,5.468803
mom_60,4358.0,-3.913039e-17,1.000000,-4.230405,-0.510683,0.038399,0.498825,7.319194
vol_5,4358.0,-1.565216e-16,0.999999,-1.141569,-0.576360,-0.258785,0.220015,9.299008
vol_20,4358.0,2.086954e-16,0.999999,-1.108561,-0.580358,-0.267557,0.167505,6.548732
vol_60,4358.0,-3.913039e-16,0.999998,-1.009511,-0.588962,-0.294532,0.141888,4.390668
ma_dist_20,4358.0,0.000000e+00,1.000000,-8.138161,-0.491339,0.075356,0.556225,6.416167
drawdown_60,4358.0,2.608693e-17,1.000000,-6.678514,-0.310223,0.324490,0.672154,0.771493
trend_strength,4358.0,-1.956520e-17,1.000000,-1.300185,-0.787181,-0.212802,0.552613,4.111470


In [35]:
df_feat["Close"] = temp_copy["Close"]

In [36]:
with open('../../data/nifty_ts2vec.pkl', 'wb') as file:
    pickle.dump(df_feat, file)